# 09주차 · 멀티모달 임베딩

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 텍스트와 이미지를 공동 임베딩 공간에 배치하는 원리를 설명한다.
- CLIP의 쌍대 인코더 구조와 대조학습을 이해한다.
- zero-shot 이미지 분류 결과를 정량·정성 평가한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
from PIL import Image, ImageDraw

def shape_image(shape, color, size=224):
    image = Image.new("RGB", (size, size), "white")
    draw = ImageDraw.Draw(image)
    if shape == "circle": draw.ellipse((45, 45, 179, 179), fill=color)
    if shape == "square": draw.rectangle((45, 45, 179, 179), fill=color)
    return image

images = [shape_image("circle", "blue"), shape_image("square", "red")]
labels = ["파란 원", "빨간 사각형"]
fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
for ax, image, label in zip(axes, images, labels): ax.imshow(image); ax.set_title(label); ax.axis("off")
plt.show()


## 선택 실습: CLIP

모델은 최초 실행 시 다운로드가 필요하다. 한국어 질의 성능이 낮으면 영문 프롬프트와 다국어 CLIP 모델을 비교하는 연구 질문으로 확장할 수 있다.


In [ ]:
RUN_CLIP = False
if RUN_CLIP:
    import torch
    from transformers import CLIPModel, CLIPProcessor
    model_name = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)
    prompts = ["a blue circle", "a red square", "a green triangle"]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).numpy()
    display(pd.DataFrame(probs, index=labels, columns=prompts))
else:
    print("CLIP 선택 실습을 건너뜁니다.")


## 학생 활동

- 도형·색상 조합 이미지를 9개 이상 생성하라.
- 텍스트 프롬프트의 구체성에 따른 순위 변화를 기록하라.
- 한국어와 영어 프롬프트 성능을 비교하라.
- 사람·직업·성별과 관련된 실제 이미지는 편향 분석 계획 없이 사용하지 않는다.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
